# ಪಾಠ 09 - ಮೆಟಾಕಾಗ್ನಿಷನ್ ವಿನ್ಯಾಸ ಮಾದರಿ


## ಸಂಯೋಜನೆ

ಈ ನೋಟ್ಪುಸ್ತಕವು Microsoft ಏಜೆಂಟ್ ಫ್ರೇಮ್ವರ್ಕ್ ಬಳಸಿಕೊಂಡು ಮೆಟಾಕಗ್ನಿಷನ್ ವಿನ್ಯಾಸ ಮಾದರಿಯನ್ನು ಪ್ರದರ್ಶಿಸುತ್ತದೆ.

**ಮುಂದಿನ ಅವಶ್ಯಕತೆಗಳು:**
- ಪರಿಸರ ಚರಗಳನ್ನು ಮೂಲಕ ಸಂರಚಿತ ಮಾಡಿದ Azure OpenAI ನಿಯೋಜನೆ
- Azure CLI ಪ್ರಾಮಾಣೀಕರಿಸಲಾಗಿದೆ (`az login`)


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## ಮೆಟಾಕಾಗ್ನಿಶನ್ ಎಂದರೆ 무엇인가?

ಮೆಟಾಕಾಗ್ನಿಶನ್ ಅಂದರೆ **ಚಿಂತನೆ ಬಗ್ಗೆ ಚಿಂತನೆ ಮಾಡುವುದು**. AI ಏಜೆಂಟ್‌ಗಳ ಸಂದರ್ಭದಲ್ಲಿದ್ದು, ಇದಕ್ಕೆ ಅರ್ಥವಾಗುತ್ತದೆ ಇದೇ ರೀತಿ ಏಜೆಂಟ್‌ಗಳನ್ನು ನಿರ್ಮಿಸುವುದು:

- ತಮ್ಮ ಸ್ವಂತ ಔಟ್‌ಪುಟ್‌ಗಳು ಮತ್ತು ಜ್ಞಾನ ಪ್ರಕ್ರಿಯೆಗಳನ್ನು **ಸ್ವಯಂ-ಪರಿಶೀಲನೆ** ಮಾಡುವುದು
- **ತಪ್ಪುಗಳನ್ನು ಪತ್ತೆಹಚ್ಚಿ** ಮೌನವಾಗಿ ವೈಫಲ್ಯವಾಗದೆ Gracefully ಪರಿಹರಿಸುವುದು
- ತಮ್ಮ ಪ್ರತಿಕ್ರಿಯೆಗಳು ಪರಿಪೂರ್ಣ ಮತ್ತು ಸಹಾಯಕಾರಿಯಾಗಿದೆಯೇ ಎಂದು **ಮೌಲ್ಯಮಾಪನ** ಮಾಡುವುದು
- ಪ್ರಾಥಮಿಕ ವಿಧಾನ ಕಾರ್ಯನಿರ್ವಹಿಸದಿದ್ದರೆ (ಉದಾ: ಬ್ಯಾಕಪ್ ವ್ಯವಸ್ಥೆಗೆ ಹಿಂತಿರುಗುವಂತೆ) ತಂತ್ರವನ್ನು **ಸ್ಥಿತಿಗತಿಯೂಗ್ಮಿಯಾಗಿ ಬದಲಿಸುವುದು**

ಮೆಟಾಕಾಗ್ನಿಟಿವ್ ಏಜೆಂಟ್ ಕೇವಲ ಪ್ರಶ್ನೆಗಳಿಗೆ ಉತ್ತರ ನೀಡುವುದಿಲ್ಲ — ಅದು ತನ್ನ ಸ್ವಂತ ಕಾರ್ಯಕ್ಷಮತೆಯನ್ನು ಮೇಲ್ವಿಚಾರಣೆ ಮಾಡುತ್ತದೆ ಮತ್ತು ತಕ್ಷಣ ಸಂಶೋಧಿಸುತ್ತಿರುವಾಗ ಸರಿಪಡಿಸುತ್ತದೆ.


## ಪ್ರಾಥಮಿಕ ಮತ್ತು ಬ್ಯಾಕಪ್ ಉಪಕರಣಗಳು

ಸಾಮಾನ್ಯ ಮೆಟಾಕಾಗ್ನಿಶನ್ ಮಾದರಿ **ಫಾಲ್ಬ್ಯಾಕ್ ತಂತ್ರ** ಆಗಿದೆ. ಏಜೆಂಟ್ ಮೊದಲು ಪ್ರಾಥಮಿಕ ಉಪಕರಣವನ್ನು ಪ್ರಯತ್ನಿಸುತ್ತಾನೆ; ಅದು ವಿಫಲವಾದರೆ (ಉದಾಹರಣೆಗೆ, 404 ದೋಷ), ಏಜೆಂಟ್ ವಿಫಲತೆಯನ್ನು ಗುರುತಿಸಿ ಪಾರದರ್ಶಕವಾಗಿ ಬ್ಯಾಕಪ್ ಉಪಕರಣಕ್ಕೆ ಸ್ವಿಚ್ ಆಗುತ್ತಾನೆ.

ಇದು ನೈಜ-ಜಗತ್ತು ವ್ಯವಸ್ಥೆಗಳನ್ನು ಪ್ರತಿಬಿಂಬಿಸುತ್ತದೆ, ಇಲ್ಲಿ ಪ್ರಾಥಮಿಕ ಸೇವೆಗಳು ಲಭ್ಯವಿಲ್ಲದಿರಬಹುದು ಮತ್ತು ಏಜೆಂಟ್‌ಗಳು ಪರ್ಯಾಯ ಮಾರ್ಗವನ್ನು ಆರಿಸುವ ಮೊದಲು ಸ್ವಯಂ-ನಿರ್ಣಯ ಮಾಡಬೇಕಾಗುತ್ತದೆ.

ಕೆಳಗೆ ನಾವು ಎರಡು ವಿಮಾನ ಬ.lookup ಟೂಲುಗಳನ್ನು ವ್ಯಾಖ್ಯಾನಿಸುತ್ತೇವೆ:
- **ಪ್ರಾಥಮಿಕ** — ಪ್ಯಾರಿಸ್, ಟೋಕಿಯೋ, ಮತ್ತು ಬಾರ्सಿಲೋನಾವನ್ನು ಹತ್ತಿರದಿಂದ ಆವರಿಸುತ್ತದೆ
- **ಬ್ಯಾಕಪ್** — ಬೆर्लಿನ್, ಸಿಡ್ನಿ ಮತ್ತು ನ್ಯೂಯಾರ್ಕ್ ಸಿಟಿಯನ್ನು ಆವರಿಸುತ್ತದೆ


In [ ]:
@tool(approval_mode="never_require")
def get_flight_times(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times for a destination (primary source)."""
    flights = {
        "Paris": "Departures: 08:00, 12:30, 17:45 — from $350",
        "Tokyo": "Departures: 11:00, 23:30 — from $890",
        "Barcelona": "Departures: 07:15, 14:00, 19:30 — from $280",
    }
    if destination in flights:
        return flights[destination]
    raise Exception(f"404: No flights found for {destination} in primary system")


@tool(approval_mode="never_require")
def get_flight_times_backup(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times from backup system (used when primary fails)."""
    backup_flights = {
        "Berlin": "Departures: 09:00, 16:00 — from $220",
        "Sydney": "Departures: 22:00 — from $1200",
        "New York City": "Departures: 06:00, 10:30, 15:00, 20:00 — from $450",
    }
    return backup_flights.get(
        destination,
        f"No flights found for {destination} in any system. Please try again later.",
    )

## ದೋಷ ಮರುಪಡೆಯುವಿಕೆ ಹೊಂದಿರುವ ಸ್ವಯಂ-ಪರಿಶೀಲಕ ಏಜೆಂಟ್

ಕೆಳಗಿನ ಏಜೆಂಟ್ ಪ್ರಾಥಮಿಕ ವಿಮಾನ ವ್ಯವಸ್ಥೆಯನ್ನು ಮೊದಲು ಪ್ರಯತ್ನಿಸಲು, ವಿಫಲತೆಯನ್ನು ಗುರುತಿಸಲು ಹಾಗೂ ಪಾರದರ್ಶಿಯಾಗಿ ಬ್ಯಾಕಪ್ ವ್ಯವಸ್ಥೆಗೆ ಹಿಂದಿರುಗಲು ಸೂಚಿಸಲಾಗಿದೆ. ಪ್ರತಿಯೊಬ್ಬ ಪ್ರತಿಕ್ರಿಯೆಯ ನಂತರ ಅದು ಸ್ವಲ್ಪಕಾಲ ಸ್ವಯಂ-ಮೌಲ್ಯಮಾಪನ ಮಾಡುತ್ತದೆ ಮತ್ತು ಅದು ಬಳಕೆದಾರನ ಪ್ರಶ್ನೆಯನ್ನು ಸಂಪೂರ್ಣವಾಗಿ ಉತ್ತರಿಸಿದೆಯೇ ಎಂದು ಪರಿಶೀಲಿಸುತ್ತದೆ.


In [ ]:
agent = client.as_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="FlightBookingAgent",
    instructions="""You are a flight booking agent with self-reflection capabilities.

When looking up flights:
1. Try the primary flight system first (get_flight_times)
2. If the primary system fails (404 error), acknowledge the error and try the backup system (get_flight_times_backup)
3. Always explain to the user what happened — be transparent about fallbacks
4. If both systems fail, apologize and suggest alternatives

After each response, briefly evaluate whether your answer was complete and helpful.""",
)

# Test with a destination in primary system
print("=== Test 1: Destination in primary system ===")
response = await agent.run(
    "What flights are available to Paris?",
    )
print(response)

# Test with a destination only in backup system
print("\n=== Test 2: Destination only in backup system ===")
response = await agent.run(
    "What flights are available to Berlin?",
    )
print(response)

## ಸ್ವ-ಮೌಲ್ಯಮಾಪನ ಮಾದರಿ

ಮೆಟಾಕಾಗ್ನಿಶನ್‌ನ ಮತ್ತೊಂದು ಮುಖಾಂತರವು **ಸ್ವ-ಮೌಲ್ಯಮಾಪನ**: ಪ್ರತ್ಯುತ್ತರವನ್ನು ಸಂಪೂರ್ಣತೆ, ಖಚಿತತೆ ಮತ್ತು ಸಹಾಯಕತೆಗಾಗಿ ಬೇರೆ ಏಜೆಂಟ್ (ಅಥವಾ ಎರಡನೇ ಬಾರಿ ಅದೇ ಏಜೆಂಟ್) ಪರಿಶೀಲಿಸುತ್ತದೆ.

ಕೆಳಗಡೆ ನಾವು ಪ್ರಯಾಣ ಏಜೆಂಟ್ ಪ್ರತ್ಯುತ್ತರಗಳನ್ನು ಮೂರು ಆಯಾಮಗಳಲ್ಲಿ ಅಂಕಮಾಡುವ `ResponseEvaluator` ಏಜೆಂಟ್ ಅನ್ನು ಸೃಷ್ಟಿಸುತ್ತೇವೆ.


In [ ]:
evaluation_agent = client.as_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="ResponseEvaluator",
    instructions="""You are a quality evaluator for travel agent responses.
Given a travel question and the agent's response, evaluate:
1. Completeness: Did it answer all parts of the question? (1-5)
2. Accuracy: Is the information correct? (1-5)
3. Helpfulness: Would a traveler find this useful? (1-5)
Provide a brief evaluation with scores and one suggestion for improvement.""",
)

# Evaluate the agent's response from Test 1
eval_prompt = f"""Question: What flights are available to Paris?
Agent Response: {response}

Please evaluate the above response."""

evaluation = await evaluation_agent.run(eval_prompt)
print("=== Self-Evaluation ===")
print(evaluation)

## ಸಾರಾಂಶ

ಈ ಪಾಠದಲ್ಲಿ ನೀವು ಮೈಕ್ರೋಸಾಫ್ಟ್ ಏಜೆಂಟ್ ಫ್ರೇಮ್ವರ್ಕ್ ಬಳಸಿಕೊಂಡು **ಮೆಟಾಕಾಗ್ನಿಟಿವ್ ಏಜೆಂಟ್‌ಗಳು** ಅನ್ನು ಹೇಗೆ ನಿರ್ಮಿಸಬಹುದು ಎಂಬುದನ್ನು ಕಲಿತಿರಿ:

- **ಸ್ವ-ಪರಿಶೀಲನೆ**: ಸ್ವಂತ ತರ್ಕವನ್ನು ನಿಗಾ ವಹಿಸುವ ಮತ್ತು ಏನಾಯಿತು ಎಂದು ಪಾರದರ್ಶಕವಾಗಿ ಸಂವಹನ ಮಾಡುವ ಏಜೆಂಟ್‌ಗಳು.
- **ಫಾಲ್ಬ್ಯಾಕ್‌ಗಳೊಂದಿಗೆ ದೋಷ ಪರಿಹಾರ**: ಏಜೆಂಟ್ ವಿಫಲತೆಗಳನ್ನು (ಉದಾ: 404 ದೋಷಗಳು) ಕಂಡುಹಿಡಿದು ಸ್ವಯಂಚಾಲಿತವಾಗಿ ಬದಲಿ ಮೂಲವನ್ನು ಪ್ರಯತ್ನಿಸುವ ಪ್ರಾಥಮಿಕ + ಬ್ಯಾಕಪ್ ಸಾಧನ ಮಾದರಿ.
- **ಸ್ವ-ಮೌಲ್ಯಮಾಪನ**: ಪೂರ್ಣತೆ, ಸರಿಯಾದತೆ ಮತ್ತು ಸಹಾಯಕತೆಗೆ ಪ್ರತಿಕ್ರಿಯೆಗಳಿಗೆ ಅಂಕಗಳನ್ನು ನೀಡುವ ಪ್ರತ್ಯೇಕ ಮೌಲ್ಯಮಾಪಕ ಏಜೆಂಟ್.

ಈ ಮಾದರಿಗಳು ಏಜೆಂಟ್‌ಗಳನ್ನು ಹೆಚ್ಚು ದృಢ, ಪಾರದರ್ಶಕ ಮತ್ತು ನಂಬಿಗಸ್ತವಾಗಿಸುವುದರಿಂದ ಉತ್ಪಾದನಾ ನೇಮಕಾತಿಗಳಿಗಾಗಿ ಅತ್ಯಂತ ಮುಖ್ಯ ಗುಣಧರ್ಮಗಳಾಗಿವೆ.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ಅಸ್ವೀಕಾರ**:
ಈ ದಸ್ತಾವೇಜು AI ಅನುವಾದ ಸೇವೆ [Co-op Translator](https://github.com/Azure/co-op-translator) ಬಳಸಿ ಅನುವಾದಿಸಲಾಗಿದೆ. ನಾವು ನಿಖರತೆಯನ್ನು ಸಾಧಿಸಲು ಪ್ರಯತ್ನಿಸುತ್ತಿದ್ದರೂ, ದಯವಿಟ್ಟು ಗಮನಿಸಿ, ಸ್ವಯಂಚಾಲಿತ ಅನುವಾದಗಳಲ್ಲಿ ದೋಷಗಳು ಅಥವಾ ಅಸಡ್ಡೆಗಳು ಇರಬಹುದು. ಮೂಲ ಭಾಷೆಯಲ್ಲಿರುವ ಮೂಲ ದಸ್ತಾವೇಜು ಪ್ರಾಮಾಣಿಕ ಮೂಲವೆಂದು ಪರಿಗಣಿಸಬೇಕು. ಪ್ರಮುಖ ಮಾಹಿತಿಗಾಗಿ, ವೃತ್ತಿಪರ ಮಾನವ ಅನುವಾದವನ್ನು ಶಿಫಾರಸು ಮಾಡಲಾಗುತ್ತದೆ. ಈ ಅನುವಾದವನ್ನು ಬಳಸುವ ಮೂಲಕ ಉಂಟಾಗುವ ಯಾವುದೇ ತಪ್ಪು ಅರ್ಥಗಳ ಅಥವಾ ತಪ್ಪು ವ್ಯಾಖ್ಯಾನಗಳ ಬಗ್ಗೆ ನಾವು ಹೊಣೆಗಾರರಲ್ಲ.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
